# 01 — Mean-Variance Optimization: Minimum Variance (GMV) & Maximum Sharpe Ratio (MSR)

GMV and MSR as Python classes (`GMV` and `MaxSharpe` in `maplab.models`), scored by the rolling backtest (`maplab.backtest`) against two benchmarks used in every notebook: equal weight (EW) and 60/40 (SPY/IEF).

## §1 Return-based methods

GMV uses only the covariance Σ; MSR also needs expected returns μ, the input one year of data estimates worst. With estimated inputs the optimizer is an error maximizer (Michaud, 1989): it overweights assets whose μ is overestimated, its weights swing with small changes in μ (Best & Grauer, 1991), and errors in μ cost far more than errors in Σ (Chopra & Ziemba, 1993). For GMV, a long-only constraint is equivalent to shrinking Σ (Jagannathan & Ma, 2003). The same error is why 1/N is hard to beat out of sample (DeMiguel, Garlappi & Uppal, 2009), and why every model here faces EW and 60/40. Later notebooks try a better μ (03), a better Σ (04) and no μ at all (05–07).

## §2 The optimization problems

Unconstrained, both portfolios have closed forms:

$$w_{\text{GMV}} = \frac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}'\Sigma^{-1}\mathbf{1}}, \qquad w_{\text{MSR}} = \frac{\Sigma^{-1}(\mu - r_f\mathbf{1})}{\mathbf{1}'\Sigma^{-1}(\mu - r_f\mathbf{1})}.$$

On this universe the unconstrained MSR takes large offsetting long and short positions (13.0× gross at end-2022; unconstrained GMV 1.6×, §4.2), so the classes solve the long-only versions:

$$\text{GMV:}\quad \min_{w}\; w'\Sigma w \quad \text{s.t.}\quad \mathbf{1}'w = 1,\; w \ge 0,$$

$$\text{MSR:}\quad \max_{w}\; \frac{(\mu - r_f\mathbf{1})'w}{\sqrt{w'\Sigma w}} \quad \text{s.t.}\quad \mathbf{1}'w = 1,\; w \ge 0.$$

**Inputs**, from the 252 trading days before each rebalance: $\Sigma$ is the annualized sample covariance of log returns; $\mu = \overline{r}_{\log} \times 252 + \tfrac12\,\mathrm{diag}(\Sigma)$ converts the log mean into an arithmetic mean; $r_f$ is the trailing mean of BIL. **Sharpe ratio** $= \text{mean}(r - r_f)\,/\,\text{std}(r - r_f)\cdot\sqrt{252}$, on daily net returns.

## Setup

The 13 ETFs, BIL as the risk-free rate and the conventions of notebook 00; log returns feed μ and Σ, simple returns compound the backtest.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import maplab as ml
import _nb01_helpers as h
from maplab import Panel, apply_style
from maplab.models import tangency_closed_form

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf = ml.load_rf_returns()                      # BIL, risk-free only
rf_daily = rf["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": rf})

print(f"prices: {prices.shape[0]:,} days × {prices.shape[1]} assets "
      f"({prices.index.min().date()} → {prices.index.max().date()})")

## §3 The classes and the rolling backtest

Calling a model at a rebalance date runs `_estimate` (μ and Σ from the prior 252 trading days), then `predict_weights` (the §2 problem, solved by the shared solvers below), then `_validate` (no missing weights, weights sum to 1, and for long-only classes no negative weights). `_estimate`, shared by every model in this repo, is printed below.

In [ ]:
import inspect
import textwrap
from maplab.models import Strategy

fn = Strategy._estimate
print(f"# ── {fn.__qualname__} " + "─" * 40)
print(textwrap.dedent(inspect.getsource(fn)))

**Solvers.** Both use SLSQP with analytic gradients (tolerance $10^{-12}$). MSR solves the convex form $\min_y y'\Sigma y$ s.t. $(\mu - r_f\mathbf{1})'y = 1$, $y \ge 0$, then sets $w = y/\mathbf{1}'y$; it falls back to GMV if no asset has $\mu_i > r_f$, and retries once from an interior start if the solve fails. GMV raises an error on failure.

**Backtest** (`ml.backtest`), identical for every model: rebalance at each month-end using only earlier data (2022-12-31 uses data through 12-30 and trades on 2023-01-03); pay 10 bp per unit of one-way turnover; let weights drift between rebalances; score from 2009-01-31, 208 rebalances in all.

## §4 Snapshot at the train/test split

One rebalance date, 2022-12-31: what the classes see and what they do with it.

### §4.1 Inputs at the snapshot

The estimation window, $r_f$, and the μ and Σ returned by `_estimate`.

In [ ]:
from maplab import GMV, MaxSharpe

rdates = ml.rebalance_dates(simple_returns.index)
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)
split_rebal = rdates[rdates <= split_ts].max()

mu, Sigma = GMV()._estimate(panel, split_rebal)
rf_ann = MaxSharpe()._rf_ann(panel, split_rebal)
window = panel.slice(split_rebal, "returns", 252)

print(f"snapshot date: {split_rebal.date()}")
print(f"estimation window: {window.index.min().date()} -> {window.index.max().date()} "
      f"({len(window)} trading days)")
print(f"rf (trailing 252-day BIL mean, annualized): {rf_ann:.2%}\n")

vol = pd.Series(np.sqrt(np.diag(Sigma.to_numpy())), index=Sigma.index)
inputs = pd.DataFrame({"mu": mu, "vol": vol, "mu - rf": mu - rf_ann}).rename_axis(None)
print(inputs.sort_values("mu - rf", ascending=False).to_string(float_format=lambda v: f"{v:.4f}"))

corr = (Sigma / np.outer(vol, vol)).rename_axis(None).rename_axis(None, axis=1)
print("\ncorrelation (252-day window):")
print(corr.to_string(float_format=lambda v: f"{v:.2f}"))

**Reading.** The window is the 2022 rate shock. Only DBC (+19.3%) and UUP (+7.7%) have μ above $r_f$; GLD sits just below it (−0.3%) and the other ten are negative. UUP also has the lowest volatility (9.27%; TIP 9.34%) and is negatively correlated with every other asset (−0.22 to −0.75). MSR allocates as if 2022's average returns continue.

### §4.2 Long-only vs unconstrained weights

The classes' long-only weights next to the closed forms of §2 on the same inputs. The tangency closed form exists only if the unconstrained GMV's expected return exceeds $r_f$, which is checked first.

In [ ]:
from maplab.models import gmv_closed_form

w_gmv = GMV()(panel, split_rebal)
w_msr = MaxSharpe()(panel, split_rebal)
w_gmv_unc = gmv_closed_form(Sigma)

mu_gmv_unc = float(w_gmv_unc @ mu)
print(f"unconstrained GMV expected return: {mu_gmv_unc:.2%} vs rf {rf_ann:.2%} "
      f"-> closed-form tangency defined: {mu_gmv_unc > rf_ann}\n")
w_tan = tangency_closed_form(mu, Sigma, rf=rf_ann)

weights = pd.DataFrame({
    "GMV long-only": w_gmv,
    "MSR long-only": w_msr,
    "GMV unconstrained": w_gmv_unc,
    "MSR unconstrained": w_tan,
}).rename_axis(None)
print(weights.sort_values("MSR unconstrained", ascending=False)
      .to_string(float_format=lambda v: f"{v:.4f}"))

The four portfolios as bars (own y-axis per panel, UUP in red), with gross leverage, total short weight and largest absolute weight below.

In [ ]:
fig, leverage_table = h.unconstrained_vs_long_only(Sigma, mu, rf_ann, w_gmv, w_msr)
plt.show()

print(leverage_table.to_string(float_format=lambda v: f"{v:.4f}"))

**Reading.** Long-only GMV holds UUP 58%, EFA 17%, IEF 14% and GLD 11%; long-only MSR holds UUP 74%, DBC 23% and GLD 3% (GLD enters as a hedge for UUP despite a slightly negative excess return). Unconstrained, MSR runs 13.0× gross leverage: +193% UUP and +165% EFA against −216% TIP and −126% EEM. $\Sigma^{-1}$ turns small gaps in μ between highly correlated assets into large offsetting bets: EFA and EEM (ρ = 0.82) differ by 7.6 points of μ and end up at +165% and −126%. Unconstrained GMV has no μ to bet on and stays at 1.6× gross. That is estimation error at full strength, and the reason the classes are long-only.

### §4.3 Sensitivity to μ

Perturb μ with draws from $N(0, s^2\Sigma)$ and re-solve MSR, 200 draws per noise level $s$. That is the sampling error of a one-year mean: the standard errors equal the annualized volatilities, and the errors are correlated like the returns. GMV does not use μ and is unchanged by construction.

In [ ]:
from maplab.models import _min_variance_long_only, _tangency_long_only

n_draws = 200
w_msr_base = w_msr.to_numpy()
Sigma_np = Sigma.to_numpy()

rows = []
for scale in (0.25, 0.5, 1.0):
    rng = np.random.default_rng(20260924)
    for _ in range(n_draws):
        eps = rng.multivariate_normal(np.zeros(len(mu)), Sigma_np)  # one-year sampling error of mu: N(0, Sigma)
        excess = mu.to_numpy() + scale * eps - rf_ann
        if excess.max() <= 0:
            continue
        w_m, _ = _tangency_long_only(excess, Sigma_np, "MSR", split_rebal)
        rows.append({
            "noise": scale,
            "half_L1": 0.5 * np.abs(w_m - w_msr_base).sum(),
            "max_w": w_m.max(),
            "holdings": int((w_m > 1e-4).sum()),
            "top": mu.index[int(np.argmax(w_m))],
        })
sens = pd.DataFrame(rows)

table = sens.groupby("noise").agg(
    draws=("half_L1", "size"),
    half_L1_median=("half_L1", "median"),
    half_L1_p90=("half_L1", lambda s: s.quantile(0.9)),
    max_w_median=("max_w", "median"),
    holdings_median=("holdings", "median"),
    UUP_top_share=("top", lambda s: (s == "UUP").mean()),
)
print("MSR weight change vs the unperturbed solution (noise in units of the one-year std error of mu):")
print(table.to_string(float_format=lambda v: f"{v:.3f}"))

gmv_change = 0.5 * np.abs(_min_variance_long_only(Sigma) - w_gmv.to_numpy()).sum()
print(f"\nGMV weight change (does not use mu): {gmv_change:.1e}")

**Reading.** At one standard error the median draw reallocates 25% of MSR's portfolio (53% at the 90th percentile), and UUP loses the top spot in 15% of draws; one draw at that level leaves no asset above $r_f$ and is skipped. At a quarter of that noise the median draw still reallocates 8%, and UUP stays on top in every draw. The median draw holds three assets at every level. This is the error maximizer of §1; with μ re-estimated every month, it is consistent with MSR's 248% turnover in §5.

### §4.4 The long-only efficient frontier

The long-only frontier on the §4.1 inputs, the capital market line through MSR, and the benchmarks. The grey cloud is 20,000 random long-only portfolios drawn from a Dirichlet(0.3); α < 1 concentrates the draws, so the cloud reaches the corners where GMV and MSR sit.

In [ ]:
mc_alpha = 0.3
fig = h.frontier_figure(Sigma, mu, w_gmv, w_msr, mc_alpha, rf_ann, split_rebal)
plt.show()

**Reading.** On 2022's inputs EW and 60/40 sit inside the cloud with negative expected returns, while MSR sits where the capital market line touches the frontier. That ranking is only as good as one year of μ; §5 shows what happened next.

### §4.5 Capital vs risk

Capital weights (dark) against each asset's share of portfolio variance (light) at 2022-12-31; a negative share is a hedge. Table columns: UUP weight; effective N $= 1/\sum_i w_i^2$; `n_uncorr`, the exponential of the entropy of the variance shares of Σ's principal portfolios (Meucci, 2009), between 1 and 13; largest variance share; diversification ratio $w'\sigma/\sigma_p$; ex-ante volatility. For long-only GMV the two bars coincide for every held asset: its optimality condition equalizes marginal risk, so each held asset's variance share equals its weight.

In [ ]:
from maplab import EqualWeight, FixedWeight

weights_snap = {
    "GMV": w_gmv,
    "MaxSharpe": w_msr,
    "EW": EqualWeight()(panel, split_rebal),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40")(panel, split_rebal),
}
print(ml.diagnostics.diversification_table(weights_snap, Sigma).to_string(float_format=lambda v: f"{v:.4f}"))

fig = ml.plotting.capital_vs_risk(weights_snap, Sigma, title=f"Capital weight vs risk share, {split_rebal.date()} (sample Σ)")
plt.show()

**Reading.** Holding many assets is not holding many bets. EW's 13 assets amount to about 1.4 uncorrelated bets, and SPY carries 89% of 60/40's variance for 60% of its capital. In EW, UUP's variance share is negative: it hedges. MSR concentrates both ways: UUP is 74% of capital and 57% of variance, DBC most of the rest. GMV has the highest diversification ratio (3.69) and about 7.6 uncorrelated bets from four assets; the count is of principal portfolios, not assets, so it can exceed the number held.

## §5 Out-of-sample results

Walk-forward: at each month-end the class re-estimates μ and Σ on the prior 252 days and holds the weights for a month, so every return is out of sample. "Train" and "test" split one track record: nothing is fitted on "train", but the project's design choices used data to 2022, so 2023 onward is the cleaner read. The cell ends with the reproduction gate: the Sharpe ratios must match `registrations/reproduction.toml` to 2 dp, the values later notebooks check against.

In [ ]:
from maplab import EqualWeight, FixedWeight, backtest, summary

strategies = {
    "GMV": GMV(),
    "MaxSharpe": MaxSharpe(),
    "EqualWeight": EqualWeight(),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40"),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag}

rows = []
for name, r in results.items():
    net = r["net"]
    turnover = r["diag"]["turnover_per_rebalance"]
    windows = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in windows.items():
        s = summary(net.loc[net_sel], turnover.loc[t_sel], rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]

msr = strategies["MaxSharpe"]
print(f"MaxSharpe fallback_dates: {len(msr.fallback_dates)}   retry_dates: {len(msr.retry_dates)}")
display_cols = ["ann_return", "ann_vol", "sharpe", "max_dd", "ann_turnover"]
print(summary_table[display_cols].rename(index={"EqualWeight": "EW"}, level=0)
      .to_string(float_format=lambda v: f"{v:.4f}"))

ml.inference.check_reproduction(summary_table, {
    "GMV": "GMV_sample", "MaxSharpe": "MaxSharpe_sample",
    "EqualWeight": "EW", "60/40": "SixtyForty_SPY_IEF",
})
print("\nreproduction gate passed: GMV, MaxSharpe, EW and 60/40 match registrations/reproduction.toml to 2 dp")

**Reading.** Over the full sample 60/40 has the highest Sharpe ratio (0.91), ahead of EW (0.74), MSR (0.72) and GMV (0.62). From 2023 MSR leads (1.13; 60/40 1.00, EW 0.91, GMV 0.76), but that window is 3.3 years and its Sharpe intervals are wide (notebook 08: 7 of 10 lower bounds below zero). In every window GMV has the lowest volatility and the shallowest drawdown and 60/40 the highest and the deepest (full sample 3.2% / −6.7% against 10.1% / −21.3%). MSR turns over 248% a year (GMV 71%, benchmarks 11–14%), yet at 10 bp that costs about 0.25% a year, against a return gap to EW of 1.2 points (5.8% vs 7.0%): costs explain little of its lag. No Sharpe gap against EW is significant; wealth paths and tests are in notebook 08.

## §6 What drove the results: allocation through time

### §6.1 Weights by asset group

Target weights summed into the six groups at every rebalance (dotted line: train/test split). EW and 60/40 are fixed and appear only in the table.

In [ ]:
gmv_group = ml.diagnostics.weights_by_group(results["GMV"]["wlog"])
msr_group = ml.diagnostics.weights_by_group(results["MaxSharpe"]["wlog"])

fig = ml.plotting.group_stackplot(results, ["GMV", "MaxSharpe"], figsize=(9, 8),
                                  title_fmt="{name} — target weights by asset group", split_ts=split_ts)
plt.show()

print("Mean target weight by asset group:")
group_means = pd.DataFrame({
    "GMV": gmv_group.mean(),
    "MaxSharpe": msr_group.mean(),
    "EW": ml.diagnostics.weights_by_group(results["EqualWeight"]["wlog"]).mean(),
    "60/40": ml.diagnostics.weights_by_group(results["60/40"]["wlog"]).mean(),
}).rename_axis(None)
print(group_means.to_string(float_format=lambda v: f"{v:.4f}"))

uup_w = results["GMV"]["wlog"]["UUP"]
print(f"\nGMV: UUP mean weight = {uup_w.mean():.4f}  (min={uup_w.min():.4f}, max={uup_w.max():.4f})")

msr_peaks = pd.DataFrame({"max weight": msr_group.max(),
                          "first date at max": msr_group.idxmax().dt.date}).rename_axis(None)
print("\nMaxSharpe: peak weight by group")
print(msr_peaks.to_string(float_format=lambda v: f"{v:.4f}"))
near_all = msr_group[msr_group.max(axis=1) >= 0.95]
print(f"\nMaxSharpe rebalances with one group >= 95%: {len(near_all)} of {len(msr_group)}")
for d, row in near_all.iterrows():
    print(f"  {d.date()}  {row.idxmax()}  {row.max():.4f}")

**Reading.** GMV is a dollar-and-bonds portfolio: 37% UUP on average (18–58%), 21% rates, 16% TIPS, 13% credit and only 7% equities, against 23% for EW and 60% for 60/40. Low volatility is its objective; the low return goes with holding so little equity. MSR's averages look diversified but hide the path: equity at 95–100% from February to April 2014 and 100% in August 2017, and single-group peaks of 85% in credit (May 2011) and the dollar (October 2022), 79% in rates (December 2020) and 61% in TIPS (May 2021). Because μ is each asset's trailing one-year mean (§2), MSR rotates into whatever had the best risk-adjusted year, with nothing to smooth the switch.

### §6.2 Concentration and turnover

Effective N, largest weight, UUP weight and trailing-12-rebalance turnover at every rebalance; EW dashed in the effective-N and turnover panels.

In [ ]:
names = ["GMV", "MaxSharpe", "60/40"]
colors = {name: ml.FAMILY_COLORS[strategies[name].family] for name in ["GMV", "MaxSharpe"]}
colors["60/40"] = ml.plotting.METHOD_COLORS["60/40"]

fig = ml.plotting.concentration_panel(
    results, names,
    rows=("effN", "max_w", "highlight", "turnover"), ref="EqualWeight", split_ts=split_ts,
    figsize=(9.5, 14), colors=colors,
    suptitle="GMV, MaxSharpe and 60/40: effN, max weight, UUP weight, turnover",
)
plt.show()

conc = pd.DataFrame({
    name: {
        "median effN": results[name]["wlog"].apply(ml.diagnostics.effective_n, axis=1).median(),
        "median max weight": results[name]["wlog"].max(axis=1).median(),
        "largest max weight": results[name]["wlog"].max(axis=1).max(),
        "ann. turnover": summary_table.loc[(name, "full"), "ann_turnover"],
    }
    for name in ["GMV", "MaxSharpe", "EqualWeight", "60/40"]
}).T.rename(index={"EqualWeight": "EW"})
print(conc.to_string(float_format=lambda v: f"{v:.3f}"))

**Reading.** Both optimizers are concentrated (median effective N 3.8 for GMV, 3.0 for MSR, against 13 for EW), but 60/40 is more concentrated than either (1.9) and has the best Sharpe (§5): a low effective N does not by itself mean a worse result. What separates GMV from MSR is stability. GMV's largest weight never exceeds 58%, while MSR's reaches 100%, and MSR pays for that with 248% annual turnover. Both converge on UUP into the 2022 split: GMV's UUP weight reaches its 58% maximum there, and MSR's fx weight its 85% peak (§6.1).

## §7 TAKEAWAY — the benchmarks win on Sharpe; the optimizers differ in risk and stability

- **The fixed-weight benchmarks are hard to beat.** Full-sample Sharpe: 60/40 0.91, EW 0.74, MSR 0.72, GMV 0.62. MSR leads from 2023 (1.13), but over 3.3 years, and no gap against EW is significant (notebook 08).
- **Risk ranks the same in every window; cost barely matters.** GMV always has the lowest volatility and 60/40 the highest (3.2% vs 10.1% full sample). MSR turns over 248% a year, which at 10 bp costs about 0.25% a year.
- **MSR is an estimation-error maximizer.** One standard error of noise in μ moves a median 25% of its weights (§4.3); with μ a trailing one-year mean, it rotates between single-group corners and reaches 100% in one asset. Unconstrained, it runs 13× gross, so long-only is not cosmetic.
- **GMV leans on the dollar.** UUP averages 37% of GMV (18–58%) with only 7% in equities; at the snapshot UUP is negatively correlated with all 12 other assets (§4.1), so it enters as a hedge. Both optimizers peak in UUP around the 2022 split (§6.2).
- **Many assets are not many bets, and few are not the problem.** EW's 13 assets amount to about 1.4 uncorrelated bets, and SPY carries 89% of 60/40's variance (§4.5); yet 60/40, the most concentrated portfolio, has the best Sharpe. MSR's weakness is instability, not concentration (§6.2).

Sample-mean μ is the weak link: notebook 03 tries a better μ, notebook 04 a better Σ, and notebooks 05–07 drop μ altogether.